# ISIC 2016 — Two-Stage Pipeline: U-Net Segmentation + Pretrained-Backbone Classification

This notebook splits the task into its two natural jobs:

1. **Stage A — Segmentation:** a proper U-Net (encoder + decoder + skip connections) trained on ISIC 2016
   **Task 1 (lesion segmentation)** data, producing a binary lesion mask for each image.
2. **Stage B — Classification:** a **pretrained CNN backbone** (EfficientNetB0, ImageNet weights) trained on
   ISIC 2016 **Task 3 (benign vs malignant)** data, where each image is first cropped/masked to the lesion
   region using the U-Net from Stage A. This "segment-then-classify" design is common in the melanoma
   literature because it removes background skin, hair, and imaging artifacts before classification.

Includes: segmentation Dice/IoU, classification accuracy/loss curves, confusion matrix, ROC-AUC, and Grad-CAM
on the classifier.

**Runtime:** Set Colab to GPU (`Runtime > Change runtime type > T4 GPU`) before running.


In [ ]:
# 1. Setup & Imports
!pip install -q tensorflow scikit-learn seaborn

import os, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, applications
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, roc_auc_score, ConfusionMatrixDisplay)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 256           # segmentation working resolution
CLS_IMG_SIZE = 224       # EfficientNetB0 native input size
BATCH_SIZE = 16
SEG_EPOCHS = 60
CLS_EPOCHS = 40
DATA_DIR = "/content/isic2016"
os.makedirs(DATA_DIR, exist_ok=True)


## 2. Download ISIC 2016 Data (Task 1: Segmentation + Task 3: Classification)

Direct downloads from the ISIC Challenge archive. If a link has moved, download manually from
https://challenge.isic-archive.com/data/#2016 and upload the zips to `/content/isic2016/`.


In [ ]:
FILES = {
    # --- Task 1: Lesion Segmentation ---
    "seg_train_images": ("ISBI2016_ISIC_Part1_Training_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_Data.zip"),
    "seg_train_masks": ("ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip"),
    "seg_test_images": ("ISBI2016_ISIC_Part1_Test_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_Data.zip"),
    "seg_test_masks": ("ISBI2016_ISIC_Part1_Test_GroundTruth.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_GroundTruth.zip"),

    # --- Task 3: Classification ---
    "cls_train_images": ("ISBI2016_ISIC_Part3_Training_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Training_Data.zip"),
    "cls_train_labels": ("ISBI2016_ISIC_Part3_Training_GroundTruth.csv",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Training_GroundTruth.csv"),
    "cls_test_images": ("ISBI2016_ISIC_Part3_Test_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Test_Data.zip"),
    "cls_test_labels": ("ISBI2016_ISIC_Part3_Test_GroundTruth.csv",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Test_GroundTruth.csv"),
}

def download(url, dest):
    if os.path.exists(dest):
        print(f"Already exists: {dest}")
        return
    print(f"Downloading {url} ...")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved to {dest}")

for key, (fname, url) in FILES.items():
    dest = os.path.join(DATA_DIR, fname)
    try:
        download(url, dest)
    except Exception as e:
        print(f"!! Could not download {fname} automatically ({e}). Upload it manually to {DATA_DIR}")

for zname in ["ISBI2016_ISIC_Part1_Training_Data.zip", "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
              "ISBI2016_ISIC_Part1_Test_Data.zip", "ISBI2016_ISIC_Part1_Test_GroundTruth.zip",
              "ISBI2016_ISIC_Part3_Training_Data.zip", "ISBI2016_ISIC_Part3_Test_Data.zip"]:
    zpath = os.path.join(DATA_DIR, zname)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath, "r") as z:
            z.extractall(DATA_DIR)
        print(f"Extracted {zname}")


## 3. Build File Lists

Segmentation masks in ISIC are usually named `<image_id>_Segmentation.png`. We pair each lesion image with its mask.


In [ ]:
def list_seg_pairs(img_dir, mask_dir):
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    pairs = []
    for f in img_files:
        stem = os.path.splitext(f)[0]
        mask_name = f"{stem}_Segmentation.png"
        mask_path = os.path.join(mask_dir, mask_name)
        if os.path.exists(mask_path):
            pairs.append((os.path.join(img_dir, f), mask_path))
    return pairs

seg_train_img_dir  = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Training_Data")
seg_train_mask_dir = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Training_GroundTruth")
seg_test_img_dir   = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Test_Data")
seg_test_mask_dir  = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Test_GroundTruth")

seg_train_pairs_all = list_seg_pairs(seg_train_img_dir, seg_train_mask_dir)
seg_test_pairs      = list_seg_pairs(seg_test_img_dir, seg_test_mask_dir)

seg_train_pairs, seg_val_pairs = train_test_split(seg_train_pairs_all, test_size=0.15, random_state=SEED)

print(f"Segmentation — Train: {len(seg_train_pairs)}  Val: {len(seg_val_pairs)}  Test: {len(seg_test_pairs)}")


## 4. Stage A — U-Net for Lesion Segmentation

In [ ]:
def decode_seg_pair(img_path, mask_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def augment_seg(img, mask):
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img); mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img); mask = tf.image.flip_up_down(mask)
    img = tf.image.random_brightness(img, 0.1)
    return img, mask

def make_seg_dataset(pairs, training=False):
    img_paths = [p[0] for p in pairs]
    mask_paths = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(decode_seg_pair, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(512, seed=SEED)
        ds = ds.map(augment_seg, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

seg_train_ds = make_seg_dataset(seg_train_pairs, training=True)
seg_val_ds   = make_seg_dataset(seg_val_pairs,   training=False)
seg_test_ds  = make_seg_dataset(seg_test_pairs,  training=False)


In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x

def build_unet_segmenter(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, 64);  p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 128);     p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 256);     p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 512);     p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 1024)

    # Decoder (with skip connections)
    u4 = layers.Conv2DTranspose(512, 2, strides=2, padding="same")(bn)
    u4 = layers.Concatenate()([u4, c4]); d4 = conv_block(u4, 512)

    u3 = layers.Conv2DTranspose(256, 2, strides=2, padding="same")(d4)
    u3 = layers.Concatenate()([u3, c3]); d3 = conv_block(u3, 256)

    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(d3)
    u2 = layers.Concatenate()([u2, c2]); d2 = conv_block(u2, 128)

    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(d2)
    u1 = layers.Concatenate()([u1, c1]); d1 = conv_block(u1, 64)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(d1)
    return models.Model(inputs, outputs, name="UNet_Segmenter")

def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]) > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

unet = build_unet_segmenter()
unet.compile(optimizer=optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=[dice_coef, iou_metric])
unet.summary()


In [ ]:
seg_callbacks = [
    callbacks.ModelCheckpoint("best_unet_segmenter.keras", monitor="val_dice_coef",
                               mode="max", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_dice_coef", mode="max", patience=10,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1),
]

seg_history = unet.fit(
    seg_train_ds,
    validation_data=seg_val_ds,
    epochs=SEG_EPOCHS,
    callbacks=seg_callbacks,
)


### Segmentation Training Curves & Test Metrics

In [ ]:
hist = seg_history.history
epochs_range = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_range, hist["loss"], label="Train Loss")
axes[0].plot(epochs_range, hist["val_loss"], label="Val Loss")
axes[0].set_title("Segmentation Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, hist["dice_coef"], label="Train Dice")
axes[1].plot(epochs_range, hist["val_dice_coef"], label="Val Dice")
axes[1].set_title("Dice Coefficient"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, hist["iou_metric"], label="Train IoU")
axes[2].plot(epochs_range, hist["val_iou_metric"], label="Val IoU")
axes[2].set_title("IoU"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("segmentation_curves.png", dpi=150)
plt.show()

test_results = unet.evaluate(seg_test_ds)
print(f"Test Loss: {test_results[0]:.4f}  Test Dice: {test_results[1]:.4f}  Test IoU: {test_results[2]:.4f}")


In [ ]:
# Visualise a few segmentation predictions
sample_imgs, sample_masks = next(iter(seg_test_ds))
pred_masks = unet.predict(sample_imgs)

n_show = min(4, sample_imgs.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for i in range(n_show):
    axes[0, i].imshow(sample_imgs[i]); axes[0, i].set_title("Image"); axes[0, i].axis("off")
    axes[1, i].imshow(sample_masks[i, ..., 0], cmap="gray"); axes[1, i].set_title("Ground Truth"); axes[1, i].axis("off")
    axes[2, i].imshow(pred_masks[i, ..., 0] > 0.5, cmap="gray"); axes[2, i].set_title("U-Net Prediction"); axes[2, i].axis("off")
plt.suptitle("U-Net Segmentation Results")
plt.tight_layout()
plt.savefig("segmentation_examples.png", dpi=150)
plt.show()


## 5. Stage B — Pretrained-Backbone Classifier on Segmented Lesions

For each classification image, we run it through the trained U-Net to get a lesion mask, then apply that mask
before feeding it to the classifier. Set `MASK_MODE` below to choose how:

- `"mask"` — zero out every background (non-lesion) pixel, keep the full frame
- `"crop"` — crop to the lesion's padded bounding box only (no pixel masking)
- `"mask_crop"` (default) — zero out background pixels **and** crop to the padded bounding box

Masking removes background skin, hair, and ruler markings more aggressively than cropping alone, which is
useful if you suspect the classifier might be latching onto spurious background cues rather than the lesion
itself (a documented failure mode in ISIC-based melanoma classifiers).


In [ ]:
def load_cls_labels(csv_path, img_folder):
    df = pd.read_csv(csv_path, header=None, names=["image_id", "label"])
    def to_binary(v):
        if isinstance(v, str):
            return 1 if v.strip().lower() == "malignant" else 0
        return int(float(v))
    df["target"] = df["label"].apply(to_binary)
    df["filepath"] = df["image_id"].apply(lambda x: os.path.join(img_folder, f"{x}.jpg"))
    df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)
    return df[["filepath", "target"]]

cls_train_csv = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part3_Training_GroundTruth.csv")
cls_test_csv  = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part3_Test_GroundTruth.csv")
cls_train_img_dir = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part3_Training_Data")
cls_test_img_dir  = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part3_Test_Data")

full_cls_train_df = load_cls_labels(cls_train_csv, cls_train_img_dir)
cls_test_df = load_cls_labels(cls_test_csv, cls_test_img_dir)
cls_train_df, cls_val_df = train_test_split(
    full_cls_train_df, test_size=0.15, stratify=full_cls_train_df["target"], random_state=SEED
)
print("Classification — Train:", len(cls_train_df), " Val:", len(cls_val_df), " Test:", len(cls_test_df))


In [ ]:
# MASK_MODE controls how the U-Net's lesion mask is applied before classification:
#   "mask"       -> zero out all non-lesion (background) pixels, keep full frame, then resize
#   "crop"       -> crop to the lesion's padded bounding box (no pixel masking), then resize
#   "mask_crop"  -> zero out background pixels AND crop to the padded bounding box (both)
MASK_MODE = "mask_crop"

def get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):
    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies
    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""
    h, w = image.shape[0], image.shape[1]
    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0
    seg_input = tf.expand_dims(seg_input, 0)
    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()
    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)

    # Upsample the mask back to the original image resolution for pixel-accurate masking
    pred_mask_full = tf.image.resize(
        pred_mask_small[..., np.newaxis], [h, w], method="nearest"
    ).numpy()[..., 0].astype(np.uint8)

    ys, xs = np.where(pred_mask_small > 0)
    lesion_found = len(xs) > 0 and len(ys) > 0

    work_image = image.numpy() if hasattr(image, "numpy") else np.array(image)

    if mode in ("mask", "mask_crop"):
        if lesion_found:
            work_image = work_image * pred_mask_full[..., np.newaxis]
        # if no lesion detected, fall back to the unmasked image rather than an all-black frame

    if mode in ("crop", "mask_crop") and lesion_found:
        x_min, x_max = xs.min() / IMG_SIZE, xs.max() / IMG_SIZE
        y_min, y_max = ys.min() / IMG_SIZE, ys.max() / IMG_SIZE
        pad_x = (x_max - x_min) * pad_frac
        pad_y = (y_max - y_min) * pad_frac
        x_min = max(0.0, x_min - pad_x); x_max = min(1.0, x_max + pad_x)
        y_min = max(0.0, y_min - pad_y); y_max = min(1.0, y_max + pad_y)

        x1, x2 = int(x_min * w), int(x_max * w)
        y1, y2 = int(y_min * h), int(y_max * h)
        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]

    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])
    return out.numpy()

def preprocess_cls_row(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    region = get_lesion_region(img, unet)
    return region

def build_cropped_array(df, cache_name):
    cache_path = os.path.join(DATA_DIR, cache_name)
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        return data["X"], data["y"]
    X = np.zeros((len(df), CLS_IMG_SIZE, CLS_IMG_SIZE, 3), dtype=np.float32)
    for i, row in enumerate(df.itertuples()):
        X[i] = preprocess_cls_row(row.filepath)
        if i % 100 == 0:
            print(f"  processed {i}/{len(df)}")
    y = df["target"].values.astype(np.float32)
    np.savez_compressed(cache_path, X=X, y=y)
    return X, y

print("Segmenting + cropping lesions for classification (cached to disk after first run)...")
X_train, y_train = build_cropped_array(cls_train_df, "cls_train_cropped.npz")
X_val,   y_val   = build_cropped_array(cls_val_df,   "cls_val_cropped.npz")
X_test,  y_test  = build_cropped_array(cls_test_df,  "cls_test_cropped.npz")
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:
def make_cls_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        def aug(img, label):
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
            img = tf.image.random_brightness(img, 0.1)
            img = tf.image.random_contrast(img, 0.9, 1.1)
            return img, label
        ds = ds.shuffle(1024, seed=SEED).map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

cls_train_ds = make_cls_dataset(X_train, y_train, training=True)
cls_val_ds   = make_cls_dataset(X_val,   y_val,   training=False)
cls_test_ds  = make_cls_dataset(X_test,  y_test,  training=False)

from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print("Class weights:", class_weight)


### Pretrained Backbone Classifier (EfficientNetB0)

Two-phase fine-tuning: first train only the new classification head with the backbone frozen, then unfreeze
the top backbone layers and fine-tune at a lower learning rate.


In [ ]:
def build_pretrained_classifier(input_shape=(CLS_IMG_SIZE, CLS_IMG_SIZE, 3)):
    base_model = applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=input_shape
    )
    base_model.trainable = False  # Phase 1: frozen backbone

    inputs = layers.Input(shape=input_shape)
    x = applications.efficientnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="predictions")(x)

    model = models.Model(inputs, outputs, name="EfficientNetB0_Classifier")
    return model, base_model

cls_model, backbone = build_pretrained_classifier()
cls_model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy",
                   metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
cls_model.summary()


### Classification Training Curves

In [ ]:
epochs_range = range(1, len(combined_hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title in zip(axes, ["accuracy", "loss", "auc"], ["Accuracy", "Loss", "AUC"]):
    ax.plot(epochs_range, combined_hist[key], label=f"Train {title}")
    ax.plot(epochs_range, combined_hist[f"val_{key}"], label=f"Val {title}")
    ax.axvline(fine_tune_start_epoch, color="gray", linestyle="--", label="Fine-tuning starts")
    ax.set_title(f"{title} over Epochs"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("classification_curves.png", dpi=150)
plt.show()


## 6. Test-Set Evaluation

In [ ]:
import numpy as np
# Phase 1: train head only (backbone frozen)
# Convert class_weight keys to integers to avoid TypeError with Keras internal range() function
# The original class_weight has keys as numpy.float32, which cannot be interpreted as integers by range()
class_weight_int_keys = {int(k): v for k, v in class_weight.items()}

phase1_callbacks = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1),
]
history_phase1 = cls_model.fit(
    cls_train_ds, validation_data=cls_val_ds,
    epochs=15, class_weight=class_weight_int_keys, callbacks=phase1_callbacks
)

# Phase 2: unfreeze top of backbone and fine-tune at low LR
backbone.trainable = True
for layer in backbone.layers[:-30]:   # keep early layers frozen, fine-tune last ~30 layers
    layer.trainable = False

cls_model.compile(optimizer=optimizers.Adam(1e-5), loss="binary_crossentropy",
                   metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

phase2_callbacks = [
    callbacks.ModelCheckpoint("best_cls_model.keras", monitor="val_auc", mode="max",
                               save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]
history_phase2 = cls_model.fit(
    cls_train_ds, validation_data=cls_val_ds,
    epochs=CLS_EPOCHS, class_weight=class_weight_int_keys, callbacks=phase2_callbacks
)

# Combine history from both phases for plotting
def concat_hist(h1, h2, key):
    return h1.history[key] + h2.history[key]

combined_hist = {k: concat_hist(history_phase1, history_phase2, k) for k in history_phase1.history.keys()}
fine_tune_start_epoch = len(history_phase1.history["loss"])

In [ ]:
test_loss, test_acc, test_auc = cls_model.evaluate(cls_test_ds)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC:      {test_auc:.4f}")

y_prob = cls_model.predict(cls_test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["benign", "malignant"]))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=["benign", "malignant"]).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("cls_confusion_matrix.png", dpi=150)
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cls_roc_curve.png", dpi=150)
plt.show()


## 7. Grad-CAM on the Pretrained-Backbone Classifier

Grad-CAM is applied to the **last convolutional layer of EfficientNetB0** (`top_conv`), showing which
regions of the *segmented lesion crop* most influenced the malignant/benign prediction.


In [ ]:
def make_gradcam_heatmap(img_array, model, backbone):
    # Layers of `model` that come after the backbone (GAP, Dropout, Dense, ..., predictions)
    backbone_idx = model.layers.index(model.get_layer(backbone.name))
    head_layers = model.layers[backbone_idx + 1:]

    with tf.GradientTape() as tape:
        x = applications.efficientnet.preprocess_input(img_array)
        conv_outputs = backbone(x, training=False)
        tape.watch(conv_outputs)

        y = conv_outputs
        for layer in head_layers:
            y = layer(y, training=False)
        predictions = y
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), predictions.numpy()[0, 0]

def overlay_gradcam(img, heatmap, alpha=0.4):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (img.shape[0], img.shape[1])).numpy()
    heatmap_resized = np.uint8(255 * heatmap_resized[..., 0])
    jet = plt.colormaps.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]
    img_norm = img / 255.0 if img.max() > 1.0 else img
    overlayed = jet_heatmap * alpha + img_norm
    overlayed = overlayed / overlayed.max()
    return overlayed

# Run Grad-CAM on a handful of test images
sample_batch_imgs, sample_batch_labels = next(iter(cls_test_ds))
n_show = min(6, sample_batch_imgs.shape[0])

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for i in range(n_show):
    img = sample_batch_imgs[i].numpy()
    true_label = "malignant" if sample_batch_labels[i].numpy() == 1 else "benign"

    img_batch = np.expand_dims(img, axis=0)
    heatmap, prob = make_gradcam_heatmap(img_batch, cls_model, backbone)
    overlay = overlay_gradcam(img, heatmap)

    pred_label = "malignant" if prob >= 0.5 else "benign"

    axes[0, i].imshow(img.astype(np.uint8))
    axes[0, i].set_title(f"True: {true_label}")
    axes[0, i].axis("off")

    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f"Pred: {pred_label} ({prob:.2f})")
    axes[1, i].axis("off")

plt.suptitle("Grad-CAM: Original Image (top) vs Model Attention Heatmap (bottom)")
plt.tight_layout()
plt.savefig("gradcam_examples.png", dpi=150)
plt.show()

# Visualisation of Results


## Notes & Next Steps

- **Why this design is stronger:** the U-Net now only has to do the job it was built for (dense pixel-wise
  segmentation), while the classifier gets a **pretrained, ImageNet-scale feature extractor** rather than
  learning visual features from ~1,300 images from scratch — this alone typically closes a large accuracy gap
  versus the baseline U-Net-encoder classifier from the previous notebook.
- **Segmentation quality matters:** if the U-Net's Dice/IoU on the Task 1 test set is low, cropping errors
  will propagate into Stage B (e.g., cropping out part of the lesion). Check the segmentation example plots
  before trusting the classifier's results.
- **Backbone options:** EfficientNetB0 is a good accuracy/speed trade-off; for a stronger baseline try
  `EfficientNetB3`/`ResNet50`/`DenseNet121` (just swap the `applications.*` call and matching `preprocess_input`).
- **Ablation worth reporting:** compare this segmented-crop classifier against training the same backbone on
  the *uncropped* full image — this quantifies how much the segmentation step actually helps, which is a
  strong result to include in your report.
